<a href="https://colab.research.google.com/github/julia-iza/Trabajo-Practico-Laboratorio/blob/main/exploracion/exploracion/exploracion_inicial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploración inicial de los datos procesados — TP Subte

Este notebook carga los datos limpios del repositorio (`data/processed/por_anio/`), los une en **un solo DataFrame** y hace la exploración inicial que pide la consigna (punto 3.1): estructura, cobertura temporal y geográfica, y calidad de los datos. Además revisa dos problemas encontrados durante el trabajo: filas donde el total no es la suma de las categorías (sección 7) y estaciones con años o meses sin datos (sección 8).

Los datos salen del repositorio de GitHub. Solo dos verificaciones opcionales (7.2 y 8.3) leen los zips originales desde Drive.

**Qué hay en los datos:** una fila por **fecha + hora + línea + estación**, con la cantidad de pasajeros que entraron por los molinetes, separados en pagos, abonados (`pax_pases_pagos`) y franquiciados (`pax_franq`). El detalle de cada columna está en `data/README.md`.

**Cómo usarlo:** Entorno de ejecución > Ejecutar todas.

**Índice:** 1. Repositorio · 2. Librerías y archivos originales · 3. Carga · 4. Tipos de datos · 5. Columnas nuevas · 6. Exploración inicial (estructura, cobertura, calidad) · 7. Total contra suma de categorías · 8. Estaciones con años o meses sin datos · 9. Resumen · 10. Uso de IA

## 1. Traer el repositorio

Si el notebook corre en Colab, clona el repositorio (o trae los últimos cambios si ya estaba clonado) y se ubica en la carpeta `notebooks/`. Así las rutas a los datos son las mismas que si se trabajara con el repo en una computadora: `../data/processed/...`.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
import os

REPO = "Trabajo-Practico-Laboratorio"

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(f"/content/{REPO}"):
        !git clone https://github.com/julia-iza/{REPO}.git /content/{REPO}
    else:
        !git -C /content/{REPO} pull
    %cd /content/{REPO}/notebooks

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 4.81 KiB | 1.60 MiB/s, done.
From https://github.com/julia-iza/Trabajo-Practico-Laboratorio
   bb09ec0..4ef6e1e  main       -> origin/main
Updating bb09ec0..4ef6e1e
Fast-forward
 estadisticos | 192 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 192 insertions(+)
 create mode 100644 estadisticos
/content/Trabajo-Practico-Laboratorio/notebooks


In [ ]:
CARPETA = "../data/processed/por_anio"
for archivo in sorted(os.listdir(CARPETA)):
    ruta = os.path.join(CARPETA, archivo)
    with open(ruta, "rb") as f:
        inicio = f.read(10)
    estado = "OK" if inicio[:2] == b"\x1f\x8b" else "NO ES GZIP"
    print(f"{archivo:30} {os.path.getsize(ruta):>12,} bytes  {estado}  {inicio!r}")

molinetes_2015.csv.gz             4,702,301 bytes  OK  b'\x1f\x8b\x08\x08\xe7\x88\xe7j\x02\xff'
molinetes_2016.csv.gz             4,962,450 bytes  OK  b'\x1f\x8b\x08\x08x\x89\xe7j\x02\xff'
molinetes_2017.csv.gz             4,991,116 bytes  OK  b'\x1f\x8b\x08\x08\x15\x8a\xe7j\x02\xff'
molinetes_2018.csv.gz             5,097,725 bytes  OK  b'\x1f\x8b\x08\x08\xb2\x8a\xe7j\x02\xff'
molinetes_2019.csv.gz             5,378,605 bytes  OK  b'\x1f\x8b\x08\x08Y\x8b\xe7j\x02\xff'
molinetes_2020.csv.gz             2,687,273 bytes  OK  b'\x1f\x8b\x08\x08\xa9\x8b\xe7j\x02\xff'
molinetes_2021.csv.gz             3,397,915 bytes  OK  b'\x1f\x8b\x08\x08\x17\x8c\xe7j\x02\xff'
molinetes_2022.csv.gz             5,064,411 bytes  OK  b'\x1f\x8b\x08\x08\xc8\x8c\xe7j\x02\xff'
molinetes_2023.csv.gz             4,995,914 bytes  OK  b'\x1f\x8b\x08\x08{\x8d\xe7j\x02\xff'
molinetes_2024.csv.gz             4,907,205 bytes  OK  b'\x1f\x8b\x08\x08!\x8e\xe7j\x02\xff'
molinetes_2025.csv.gz             5,125,681 bytes  O

## 2. Librerías y archivos originales

- Las librerías de la materia: `pandas` para las tablas, `numpy` para cálculos y `matplotlib` y `seaborn` para los gráficos. `zipfile` se usa solo para leer los originales.
- **Archivos originales (opcional):** las verificaciones 7.2 y 8.3 leen los zips de SBASE desde `Mi unidad/TP-Subte/zips/`. Con `USAR_ORIGINALES = True` se monta Drive; si los zips no están, esas celdas se saltean solas.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import zipfile

pd.set_option("display.max_columns", 20)   # mostrar todas las columnas al imprimir

USAR_ORIGINALES = True
CARPETA_ZIPS = "/content/drive/MyDrive/TP-Subte/zips"

if USAR_ORIGINALES and 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')

HAY_ORIGINALES = USAR_ORIGINALES and os.path.exists(CARPETA_ZIPS)
print("Archivos originales disponibles:", HAY_ORIGINALES)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivos originales disponibles: True


## 3. Cargar los archivos y unirlos

Hay un archivo comprimido por año en `data/processed/por_anio/`. Cada uno se lee con `pd.read_csv` (pandas descomprime el `.gz` solo) y después se unen todos con `pd.concat`, uno debajo del otro.

Se imprime cuántas filas tiene cada año para ver que se haya leído todo.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
CARPETA = "../data/processed/por_anio/"
archivos = sorted(os.listdir(CARPETA))
partes = []
for archivo in archivos:
    df_anio = pd.read_csv(os.path.join(CARPETA, archivo))
    print(f"{archivo}: {len(df_anio):,} filas")
    partes.append(df_anio)

datos = pd.concat(partes, ignore_index=True)
print(f"\nTotal: {len(datos):,} filas")

molinetes_2015.csv.gz: 522,807 filas
molinetes_2016.csv.gz: 545,897 filas
molinetes_2017.csv.gz: 547,713 filas
molinetes_2018.csv.gz: 561,732 filas
molinetes_2019.csv.gz: 589,004 filas
molinetes_2020.csv.gz: 359,496 filas
molinetes_2021.csv.gz: 426,415 filas
molinetes_2022.csv.gz: 597,791 filas
molinetes_2023.csv.gz: 583,852 filas
molinetes_2024.csv.gz: 579,277 filas
molinetes_2025.csv.gz: 585,316 filas
molinetes_2026.csv.gz: 397,591 filas

Total: 6,296,891 filas


## 4. Ajustar los tipos de datos

- `fecha` se lee como texto: se pasa a fecha con `pd.to_datetime` para poder sacar el año, el mes o el día de la semana.
- `linea` y `estacion` se repiten millones de veces. Pasarlas a tipo `category` hace que pandas guarde cada nombre una sola vez, y el DataFrame ocupa mucha menos memoria.

Se muestra la memoria antes y después del cambio.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
print(f"Memoria antes: {datos.memory_usage(deep=True).sum() / 1_000_000:,.0f} MB")

datos["fecha"] = pd.to_datetime(datos["fecha"])
datos["linea"] = datos["linea"].astype("category")
datos["estacion"] = datos["estacion"].astype("category")

print(f"Memoria después: {datos.memory_usage(deep=True).sum() / 1_000_000:,.0f} MB")

Memoria antes: 1,342 MB
Memoria después: 315 MB


## 5. Columnas nuevas para analizar

A partir de la fecha se crean columnas que sirven para agrupar:

- `anio` y `mes`;
- `periodo`: año y mes juntos (por ejemplo, `2019-04`), para series mensuales;
- `dia_semana`: 0 = lunes … 6 = domingo, y `nombre_dia` con el nombre;
- `tipo_dia`: "hábil" de lunes a viernes y "fin de semana" sábado y domingo. **No distingue feriados**: un feriado que cae un martes cuenta como hábil.

`nombre_dia` y `tipo_dia` también se pasan a `category` para ahorrar memoria. Esta celda tarda unos segundos porque `apply` recorre millones de filas.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
NOMBRES_DIAS = {0: "lunes", 1: "martes", 2: "miércoles", 3: "jueves",
                4: "viernes", 5: "sábado", 6: "domingo"}

datos["anio"] = datos["fecha"].dt.year
datos["mes"] = datos["fecha"].dt.month
datos["periodo"] = datos["fecha"].dt.to_period("M")
datos["dia_semana"] = datos["fecha"].dt.dayofweek
datos["nombre_dia"] = datos["dia_semana"].apply(lambda d: NOMBRES_DIAS[d]).astype("category")
datos["tipo_dia"] = datos["dia_semana"].apply(lambda d: "hábil" if d < 5 else "fin de semana").astype("category")


,fecha,hora,linea,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,anio,mes,periodo,dia_semana,nombre_dia,tipo_dia
0,2015-01-01,5,LineaA,PLAZA MISERERE,0,0,0,0,2015,1,2015-01,3,jueves,hábil
1,2015-01-01,5,LineaD,CATEDRAL,0,0,0,0,2015,1,2015-01,3,jueves,hábil
2,2015-01-01,5,LineaD,CONGRESO DE TUCUMAN,0,0,0,0,2015,1,2015-01,3,jueves,hábil
3,2015-01-01,5,LineaH,CASEROS,0,0,0,0,2015,1,2015-01,3,jueves,hábil
4,2015-01-01,6,LineaC,INDEPENDENCIA,0,0,0,0,2015,1,2015-01,3,jueves,hábil


## 6. Exploración inicial

### 6.1 Estructura

Dimensiones, columnas y tipo de dato de cada una. Las primeras filas se mostraron al final de la sección 5.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
print("Filas y columnas:", datos.shape)
datos.info()

Filas y columnas: (6296891, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6296891 entries, 0 to 6296890
Data columns (total 14 columns):
 #   Column           Dtype         
---  ------           -----         
 0   fecha            datetime64[ns]
 1   hora             int64         
 2   linea            category      
 3   estacion         category      
 4   pax_pagos        int64         
 5   pax_pases_pagos  int64         
 6   pax_franq        int64         
 7   total            int64         
 8   anio             int32         
 9   mes              int32         
 10  periodo          period[M]     
 11  dia_semana       int32         
 12  nombre_dia       category      
 13  tipo_dia         category      
dtypes: category(4), datetime64[ns](1), int32(3), int64(5), period[M](1)
memory usage: 432.4 MB


Últimas filas: confirman que los datos llegan hasta el 31 de agosto de 2026.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
datos.tail()

,fecha,hora,linea,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,anio,mes,periodo,dia_semana,nombre_dia,tipo_dia
6296886,2026-08-31,23,LineaH,LAS HERAS,13,0,1,14,2026,8,2026-08,0,lunes,hábil
6296887,2026-08-31,23,LineaH,ONCE,9,0,2,11,2026,8,2026-08,0,lunes,hábil
6296888,2026-08-31,23,LineaH,PATRICIOS,8,0,1,9,2026,8,2026-08,0,lunes,hábil
6296889,2026-08-31,23,LineaH,SANTA FE,12,0,5,17,2026,8,2026-08,0,lunes,hábil
6296890,2026-08-31,23,LineaH,VENEZUELA,11,0,11,22,2026,8,2026-08,0,lunes,hábil


Primeras filas: confirman que los datos comienzan desde el 1 de Enero de 2015

In [ ]:
datos.tail()

,fecha,hora,linea,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,anio,mes,periodo,dia_semana,nombre_dia,tipo_dia
6296886,2026-08-31,23,LineaH,LAS HERAS,13,0,1,14,2026,8,2026-08,0,lunes,hábil
6296887,2026-08-31,23,LineaH,ONCE,9,0,2,11,2026,8,2026-08,0,lunes,hábil
6296888,2026-08-31,23,LineaH,PATRICIOS,8,0,1,9,2026,8,2026-08,0,lunes,hábil
6296889,2026-08-31,23,LineaH,SANTA FE,12,0,5,17,2026,8,2026-08,0,lunes,hábil
6296890,2026-08-31,23,LineaH,VENEZUELA,11,0,11,22,2026,8,2026-08,0,lunes,hábil


### 6.2 Cobertura temporal y geográfica

Qué período y qué líneas cubren los datos, y para cada año cuántos días tienen datos, cuántas líneas y estaciones aparecen y cuántos pasajeros suman. Si un año tiene muchas más estaciones que los demás, puede haber nombres que no se unificaron en la limpieza.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
print("Primera fecha:", datos["fecha"].min().date())
print("Última fecha: ", datos["fecha"].max().date())
print("Días con datos:", datos["fecha"].nunique())
print("Líneas:", list(datos["linea"].cat.categories))

datos.groupby("anio").agg(dias=("fecha", "nunique"),
                          lineas=("linea", "nunique"),
                          estaciones=("estacion", "nunique"),
                          pasajeros=("total", "sum"))

Primera fecha: 2015-01-01
Última fecha:  2026-08-31
Días con datos: 4254
Líneas: ['LineaA', 'LineaB', 'LineaC', 'LineaD', 'LineaE', 'LineaH', 'LineaPM']


,dias,lineas,estaciones,pasajeros
anio,,,,
2015,365,6,82,282519120
2016,366,6,83,314418191
2017,365,6,83,328701725
2018,365,6,84,348400114
2019,362,6,86,341340501
2020,366,6,85,77504971
2021,364,6,86,96077158
2022,365,6,86,231459457
2023,364,6,86,243225292


### 6.3 Calidad

- **Nulos:** después de la limpieza no debería quedar ninguno.
- **Duplicados:** no debería haber dos filas con la misma fecha, hora, línea y estación, porque la limpieza las sumó.
- **Coherencia del total:** `total` debería ser igual a `pax_pagos + pax_pases_pagos + pax_franq`. Se analiza en detalle en la sección 7.
- **`describe()`:** resumen de las columnas numéricas. Mirar que no haya pasajeros negativos y que las horas vayan de 0 a 23.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
print("Nulos por columna:")
print(datos.isnull().sum())

duplicados = datos.duplicated(subset=["fecha", "hora", "linea", "estacion"]).sum()
print(f"\nFilas duplicadas: {duplicados:,}")

suma = datos["pax_pagos"] + datos["pax_pases_pagos"] + datos["pax_franq"]
print(f"Filas donde el total no es la suma de las categorías: {(suma != datos['total']).sum():,}")

Nulos por columna:
fecha              0
hora               0
linea              0
estacion           0
pax_pagos          0
pax_pases_pagos    0
pax_franq          0
total              0
anio               0
mes                0
periodo            0
dia_semana         0
nombre_dia         0
tipo_dia           0
dtype: int64

Filas duplicadas: 0
Filas donde el total no es la suma de las categorías: 275


Resumen estadístico de las columnas numéricas: no debería haber pasajeros negativos y las horas deberían ir de 0 a 23. Como cada fila es una estación en una hora, los valores máximos corresponden a las estaciones más concurridas en hora pico.

🤖 **Asistencia de IA:** sí — prompt E1 (ver sección 10).

In [ ]:
datos[["hora", "pax_pagos", "pax_pases_pagos", "pax_franq", "total"]].describe()

,hora,pax_pagos,pax_pases_pagos,pax_franq,total
count,6.296891e+06,6.296891e+06,6.296891e+06,6.296891e+06,6.296891e+06
mean,1.388131e+01,4.241852e+02,1.616000e+00,1.968445e+01,4.454857e+02
std,5.261124e+00,6.599467e+02,4.314120e+00,2.835677e+01,6.805347e+02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,9.000000e+00,8.000000e+01,0.000000e+00,3.000000e+00,8.500000e+01
50%,1.400000e+01,2.210000e+02,0.000000e+00,1.000000e+01,2.350000e+02
75%,1.800000e+01,5.360000e+02,1.000000e+00,2.500000e+01,5.670000e+02
max,2.300000e+01,1.783400e+04,2.930000e+02,1.245900e+04,1.798000e+04


## 7. ¿El total es la suma de las categorías?

La columna `total` debería ser igual a `pax_pagos + pax_pases_pagos + pax_franq`: según la Dirección General de Estadística y Censos, los pasajeros totales son la suma de los pagos, los franquiciados y los abonados (ver `data/README.md`). La sección 6.3 mostró cuántas filas no cumplen; acá se busca **dónde están** y **cuánto pesan**.

### 7.1 Dónde están las diferencias

🤖 **Asistencia de IA:** sí — prompt E2 y E3 (ver sección 10).

In [ ]:
suma_categorias = datos["pax_pagos"] + datos["pax_pases_pagos"] + datos["pax_franq"]
diferencia = datos["total"] - suma_categorias

total_filas = len(datos)
incorrectas = (diferencia != 0).sum()
print(f"Filas analizadas:            {total_filas:,}")
print(f"Filas con total distinto:    {incorrectas:,} ({incorrectas / total_filas * 100:.4f}%)")

malas = datos[diferencia != 0].copy()
malas["diferencia"] = diferencia[diferencia != 0]

por_anio = malas.groupby("anio").agg(filas=("diferencia", "size"),
                                     diferencia_pasajeros=("diferencia", "sum"))
por_anio["pasajeros_del_anio"] = datos.groupby("anio")["total"].sum()
por_anio["porcentaje_del_anio"] = (por_anio["diferencia_pasajeros"] / por_anio["pasajeros_del_anio"] * 100).round(4)
print("\nPor año:")
print(por_anio)

print("\nPor mes:")
print(malas.groupby("periodo").size())

print("\nPor línea y estación:")
print(malas.groupby(["linea", "estacion"], observed=True).size())

Filas analizadas:            6,296,891
Filas con total distinto:    275 (0.0044%)

Por año:
      filas  diferencia_pasajeros  pasajeros_del_anio  porcentaje_del_anio
anio                                                                      
2026    275                   612           133525596               0.0005

Por mes:
periodo
2026-06    275
Freq: M, dtype: int64

Por línea y estación:
linea    estacion
LineaPM  COCHEPM     275
dtype: int64


Las 10 filas con mayor diferencia.

🤖 **Asistencia de IA:** sí — prompt E2 y E3 (ver sección 10).

In [ ]:
malas.sort_values("diferencia", ascending=False).head(10)

,fecha,hora,linea,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,anio,mes,periodo,dia_semana,nombre_dia,tipo_dia,diferencia
6145808,2026-06-04,17,LineaPM,COCHEPM,532,3,140,684,2026,6,2026-06,3,jueves,hábil,9
6140517,2026-06-01,17,LineaPM,COCHEPM,467,3,122,600,2026,6,2026-06,0,lunes,hábil,8
6143951,2026-06-03,16,LineaPM,COCHEPM,480,3,118,608,2026,6,2026-06,2,miércoles,hábil,7
6152526,2026-06-08,16,LineaPM,COCHEPM,403,4,104,518,2026,6,2026-06,0,lunes,hábil,7
6178591,2026-06-23,19,LineaPM,COCHEPM,237,5,74,323,2026,6,2026-06,1,martes,hábil,7
6144129,2026-06-03,18,LineaPM,COCHEPM,467,3,143,620,2026,6,2026-06,2,miércoles,hábil,7
6144040,2026-06-03,17,LineaPM,COCHEPM,508,4,113,632,2026,6,2026-06,2,miércoles,hábil,7
6169850,2026-06-18,16,LineaPM,COCHEPM,487,2,128,624,2026,6,2026-06,3,jueves,hábil,7
6157881,2026-06-11,16,LineaPM,COCHEPM,464,3,131,605,2026,6,2026-06,3,jueves,hábil,7
6157972,2026-06-11,17,LineaPM,COCHEPM,512,3,97,619,2026,6,2026-06,3,jueves,hábil,7


**Qué muestra.** Todas las filas con diferencia están en **junio de 2026**, en la **línea PM** y en `COCHEPM`: no es una estación sino el conjunto de validadoras ubicadas **dentro de los coches del Premetro**. En esas filas, `total` tiene 1 o 2 pasajeros más que la suma de las categorías. La diferencia es una fracción ínfima de los pasajeros del año.

Un error de la limpieza aparecería repartido en muchos años y archivos; un problema tan concentrado apunta al archivo original. Se verifica en 7.2.

### 7.2 Verificación en el archivo original

Se lee el archivo original de junio de 2026 de la línea PM directamente desde el zip, sin pasar por la limpieza, y se buscan las filas donde el total no cierra. Cada línea del archivo viene entera entre comillas y separada por `;`, así que se lee como texto y se separa a mano.

Necesita los zips en Drive (sección 2). Si no están, la celda lo avisa y no hace nada.

🤖 **Asistencia de IA:** sí — prompt E2 y E3 (ver sección 10).

In [ ]:
if HAY_ORIGINALES:
    ARCHIVO = "Molinetes 2026/202606_PAX15min-PM-INCLUYEOTROMODOSDEPAGO.csv"
    with zipfile.ZipFile(os.path.join(CARPETA_ZIPS, "molinetes-2026.zip")) as zf:
        with zf.open(ARCHIVO) as f:
            texto = f.read().decode("utf-8-sig", errors="replace")   # utf-8-sig quita el BOM

    lineas_texto = [l.strip().strip('"') for l in texto.splitlines() if l.strip() != ""]
    filas = [l.split(";") for l in lineas_texto]
    original = pd.DataFrame(filas[1:], columns=filas[0])

    for c in ["pax_pagos", "pax_pases_pagos", "pax_franq", "pax_TOTAL"]:
        original[c] = pd.to_numeric(original[c], errors="coerce")
    original["diferencia"] = original["pax_TOTAL"] - (original["pax_pagos"] + original["pax_pases_pagos"] + original["pax_franq"])
    malas_original = original[original["diferencia"] != 0]

    print(f"Registros de 15 minutos en el original: {len(original):,}")
    print(f"Registros que no cierran:               {len(malas_original):,}")
    print(f"Diferencia en pasajeros:                {malas_original['diferencia'].sum():,.0f}")
    print("\nEstaciones:", list(malas_original["ESTACION"].unique()))
    print("Ejemplos de molinetes:", list(malas_original["MOLINETE"].unique()[:5]))
else:
    print("Los archivos originales no están disponibles: se saltea la verificación.")

Registros de 15 minutos en el original: 36,558
Registros que no cierran:               542
Diferencia en pasajeros:                612

Estaciones: ['CochePM']
Ejemplos de molinetes: ['LineaPrem_Coche7_Val3', 'LineaPrem_Coche17_Val4', 'LineaPrem_Coche9_Val4', 'LineaPrem_Coche1_Val3', 'LineaPrem_Coche10_Val4']


**Conclusión.** El archivo original tiene registros de 15 minutos de las validadoras a bordo (`LineaPrem_CocheN_ValN`) con **la misma diferencia total** que los datos procesados. La limpieza agrupa los cuatro tramos de cada hora, por eso hay más registros en el original que filas en los procesados. El problema viene de los datos de SBASE; la limpieza no lo generó.

Una posible explicación, que no se puede comprobar con estos datos, es que en esas validadoras el total incluya algún medio de pago que no entra en ninguna de las tres categorías: desde diciembre de 2024 los archivos dicen "INCLUYEOTROMODOSDEPAGO".

**Decisión:**
- Para contar pasajeros se usa `total` sin modificar.

## 8. Estaciones con años o meses sin datos

Al revisar la unificación de nombres en la limpieza apareció que **Plaza de Mayo (línea A) no tiene ningún pasajero en 2020**, y que algunas estaciones tienen bastantes menos pasajeros en ciertos años que en los vecinos. Esta sección lo busca de forma sistemática para todas las estaciones.

### 8.1 Años con pocos días de datos

Para cada estación y cada año se cuentan los días con datos y se comparan con los días con datos de toda la red ese año. Se marcan:
- los años **sin ningún dato** que están entre el primer y el último año en que la estación aparece (un hueco en el medio);
- los años con datos en **menos del 90 %** de los días.

El primer año de una estación nueva suele aparecer marcado porque se inauguró a mitad de año: eso no es un faltante.

🤖 **Asistencia de IA:** sí — prompt E4 (ver sección 10).

In [ ]:
dias_red = datos.groupby("anio")["fecha"].nunique()
dias_estacion = datos.groupby(["linea", "estacion", "anio"], observed=True)["fecha"].nunique().unstack("anio")
proporcion = dias_estacion.div(dias_red, axis=1)

huecos = []
for (linea, estacion), fila in proporcion.iterrows():
    anios_con_datos = fila.dropna().index
    for anio in proporcion.columns:
        if anio < anios_con_datos.min() or anio > anios_con_datos.max():
            continue                      # antes de que aparezca o después de que deje de aparecer
        p = fila[anio]
        if pd.isnull(p):
            huecos.append({"linea": linea, "estacion": estacion, "anio": anio,
                           "dias_con_datos": 0, "porcentaje_de_dias": 0.0})
        elif p < 0.9:
            huecos.append({"linea": linea, "estacion": estacion, "anio": anio,
                           "dias_con_datos": int(dias_estacion.loc[(linea, estacion), anio]),
                           "porcentaje_de_dias": round(p * 100, 1)})

tabla_huecos = pd.DataFrame(huecos)
print(f"Estaciones y años marcados: {len(tabla_huecos)}")
tabla_huecos

Estaciones y años marcados: 138


,linea,estacion,anio,dias_con_datos,porcentaje_de_dias
0,LineaA,ACOYTE,2023,302,83.0
1,LineaA,ALBERTI,2020,100,27.3
2,LineaA,ALBERTI,2021,136,37.4
3,LineaA,CARABOBO,2020,114,31.1
4,LineaA,CARABOBO,2021,191,52.5
...,...,...,...,...,...
133,LineaH,VENEZUELA,2021,185,50.8
134,LineaPM,COCHEPM,2025,61,16.7
135,LineaPM,COCHEPM,2026,149,61.3
136,LineaPM,SAGUIER,2025,61,16.7


**Qué mirar:**
- **0 días en un año intermedio** (como Plaza de Mayo en 2020): la estación existía antes y después, pero ese año no tiene datos. Puede ser un cierre de todo el año o un faltante en el archivo original; la sección 8.3 lo revisa.
- **Pocos días en el primer año** de una estación nueva: inauguración a mitad de año.
- **Muy pocos días en un solo año** (por ejemplo, `LORIA` en líneas que no son la A, o `ND`): registros sueltos mal asignados en el original. Suman muy pocos pasajeros.
- **Un porcentaje intermedio en una estación conocida**: posible cierre temporal. La sección 8.2 muestra en qué meses.

### 8.2 Detalle mes a mes

`detalle_estacion` muestra, para una estación, los días con datos y los pasajeros (en miles) de cada mes de cada año. Sirve para ver si un faltante es un año entero, algunos meses (por ejemplo, un cierre por obras) o una caída de pasajeros con todos los días presentes.

Se aplica a Plaza de Mayo (línea A) y a Agüero (línea D), que en la limpieza mostraron años con muchos menos pasajeros que los vecinos. Para revisar otra estación de la tabla de 8.1, alcanza con llamar a la función con su línea y su nombre.

🤖 **Asistencia de IA:** sí — prompt E4 (ver sección 10).

In [ ]:
def detalle_estacion(linea, estacion):
    seleccion = datos[(datos["linea"] == linea) & (datos["estacion"] == estacion)]
    dias = seleccion.groupby(["anio", "mes"])["fecha"].nunique().unstack("mes")
    pasajeros = (seleccion.groupby(["anio", "mes"])["total"].sum() / 1000).round(0).unstack("mes")
    return dias, pasajeros

**Plaza de Mayo (línea A).**

🤖 **Asistencia de IA:** sí — prompt E4 (ver sección 10).

In [ ]:
dias, pasajeros = detalle_estacion("LineaA", "PLAZA DE MAYO")
print("PLAZA DE MAYO (línea A): días con datos por mes")
display(dias)
print("PLAZA DE MAYO (línea A): pasajeros por mes, en miles")
display(pasajeros)

PLAZA DE MAYO (línea A): días con datos por mes


mes,1,2,3,4,5,6,7,8,9,10,11,12
anio,,,,,,,,,,,,
2015,31.0,28.0,30.0,30.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2016,31.0,29.0,31.0,30.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2017,31.0,28.0,31.0,29.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,29.0
2018,31.0,28.0,30.0,30.0,31.0,29.0,9.0,4.0,1.0,5.0,16.0,30.0
2019,31.0,28.0,31.0,29.0,29.0,30.0,31.0,31.0,30.0,31.0,29.0,1.0
2021,NaN,NaN,NaN,NaN,1.0,2.0,23.0,31.0,29.0,31.0,29.0,31.0
2022,31.0,28.0,31.0,30.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2023,31.0,28.0,29.0,27.0,31.0,30.0,31.0,30.0,30.0,31.0,30.0,31.0
2024,31.0,29.0,31.0,30.0,30.0,30.0,31.0,31.0,30.0,30.0,30.0,31.0


PLAZA DE MAYO (línea A): pasajeros por mes, en miles


mes,1,2,3,4,5,6,7,8,9,10,11,12
anio,,,,,,,,,,,,
2015,386.0,364.0,442.0,494.0,475.0,515.0,518.0,491.0,537.0,529.0,486.0,406.0
2016,402.0,393.0,482.0,500.0,514.0,483.0,479.0,537.0,531.0,511.0,507.0,443.0
2017,444.0,390.0,532.0,447.0,528.0,516.0,500.0,531.0,519.0,541.0,525.0,437.0
2018,445.0,371.0,448.0,466.0,433.0,453.0,123.0,0.0,0.0,0.0,50.0,332.0
2019,370.0,367.0,405.0,412.0,478.0,401.0,461.0,464.0,455.0,505.0,451.0,0.0
2021,NaN,NaN,NaN,NaN,0.0,0.0,49.0,100.0,134.0,159.0,185.0,200.0
2022,132.0,168.0,247.0,272.0,291.0,297.0,307.0,330.0,335.0,314.0,336.0,279.0
2023,228.0,245.0,326.0,283.0,311.0,321.0,344.0,357.0,354.0,352.0,371.0,330.0
2024,308.0,270.0,312.0,290.0,305.0,242.0,323.0,329.0,319.0,306.0,288.0,283.0


**Agüero (línea D).**

🤖 **Asistencia de IA:** sí — prompt E4 (ver sección 10).

In [ ]:
dias, pasajeros = detalle_estacion("LineaD", "AGUERO")
print("AGÜERO (línea D): días con datos por mes")
display(dias)
print("AGÜERO (línea D): pasajeros por mes, en miles")
display(pasajeros)

AGÜERO (línea D): días con datos por mes


mes,1,2,3,4,5,6,7,8,9,10,11,12
anio,,,,,,,,,,,,
2015,31.0,28.0,30.0,30.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2016,29.0,29.0,31.0,30.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2017,31.0,28.0,31.0,29.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2018,31.0,28.0,31.0,30.0,31.0,29.0,31.0,31.0,29.0,31.0,29.0,30.0
2019,31.0,28.0,31.0,29.0,29.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2020,31.0,29.0,20.0,NaN,NaN,1.0,1.0,NaN,1.0,NaN,3.0,2.0
2021,3.0,NaN,2.0,7.0,3.0,NaN,10.0,11.0,26.0,31.0,29.0,31.0
2022,31.0,28.0,31.0,30.0,31.0,30.0,31.0,31.0,30.0,31.0,30.0,31.0
2023,31.0,28.0,31.0,30.0,31.0,30.0,31.0,30.0,30.0,30.0,29.0,31.0


AGÜERO (línea D): pasajeros por mes, en miles


mes,1,2,3,4,5,6,7,8,9,10,11,12
anio,,,,,,,,,,,,
2015,176.0,188.0,240.0,275.0,278.0,288.0,302.0,291.0,308.0,304.0,289.0,246.0
2016,177.0,200.0,277.0,304.0,319.0,313.0,307.0,345.0,353.0,340.0,327.0,254.0
2017,197.0,196.0,312.0,284.0,322.0,313.0,304.0,331.0,322.0,326.0,324.0,267.0
2018,217.0,220.0,310.0,315.0,317.0,305.0,321.0,337.0,302.0,326.0,302.0,267.0
2019,221.0,230.0,285.0,285.0,312.0,279.0,308.0,323.0,312.0,331.0,299.0,273.0
2020,217.0,227.0,147.0,NaN,NaN,0.0,0.0,NaN,0.0,NaN,0.0,0.0
2021,0.0,NaN,0.0,0.0,0.0,NaN,0.0,0.0,67.0,107.0,123.0,132.0
2022,86.0,117.0,178.0,198.0,203.0,222.0,228.0,243.0,238.0,223.0,229.0,182.0
2023,126.0,155.0,315.0,307.0,318.0,269.0,219.0,230.0,232.0,228.0,237.0,198.0


**Interpretación (completar con lo que muestren las tablas):**
- Plaza de Mayo: ¿falta todo 2020 o solo algunos meses? ¿Qué pasa en 2018?
- Agüero: ¿en 2024 faltan meses o hay todos los días pero con menos pasajeros?

Las tablas no dicen **por qué** falta un período. Si un cierre por obras explica un hueco, conviene citar la fuente (por ejemplo, un comunicado de Emova o de SBASE).

### 8.3 ¿Plaza de Mayo falta en el archivo original de 2020?

Si la estación no aparece en el original, el hueco viene de los datos de SBASE y no de la limpieza. Se lee el archivo original de 2020 (solo las columnas de línea, estación y total) y se listan las estaciones de la línea A, y cualquier estación cuyo nombre contenga "mayo".

Necesita los zips en Drive (sección 2). Tarda alrededor de un minuto.

🤖 **Asistencia de IA:** sí — prompt E4 (ver sección 10).

In [ ]:
if HAY_ORIGINALES:
    with zipfile.ZipFile(os.path.join(CARPETA_ZIPS, "molinetes-2020.zip")) as zf:
        crudo_2020 = pd.read_csv(zf.open("historico2.csv"), usecols=["LINEA", "ESTACION", "pax_TOTAL"])

    print("Estaciones de la línea A en el original de 2020 (pasajeros):")
    print(crudo_2020[crudo_2020["LINEA"] == "LineaA"].groupby("ESTACION")["pax_TOTAL"].sum())

    print("\nEstaciones cuyo nombre contiene 'mayo':")
    print(crudo_2020[crudo_2020["ESTACION"].str.contains("mayo", case=False, na=False)]
          .groupby(["LINEA", "ESTACION"])["pax_TOTAL"].sum())
else:
    print("Los archivos originales no están disponibles: se saltea la verificación.")

Estaciones de la línea A en el original de 2020 (pasajeros):
ESTACION
Acoyte            1059941.0
Alberti            282295.0
Carabobo           635706.0
Castro Barros      728067.0
Congreso           992997.0
Flores            1034205.0
Lima               791499.0
Loria              442772.0
Pasco              180542.0
Peru              1770895.0
Piedras            465757.0
Plaza Miserere    1123537.0
Primera Junta     1128932.0
Puan               369927.0
Rio de Janeiro     550901.0
Saenz Peña              3.0
Saenz Pe�a         484650.0
San Pedrito       2316696.0
Name: pax_TOTAL, dtype: float64

Estaciones cuyo nombre contiene 'mayo':
LINEA   ESTACION       
LineaC  Avenida de Mayo    398204.0
Name: pax_TOTAL, dtype: float64


**Interpretación (completar con lo que muestre la salida):**
- Si Plaza de Mayo **no aparece** en la línea A del original: el hueco viene de los datos de SBASE. Se anota como limitación y, para comparar estaciones entre años, se tiene en cuenta que 2020 no incluye esa estación.
- Si **aparece con otro nombre**: hay que agregar ese nombre a `CORRECCIONES_ESTACIONES` en `notebooks/inspeccion_y_limpieza.ipynb` y volver a procesar.

## 9. Resumen de la exploración



- **Tamaño y cobertura:**filas, una por fecha, hora, línea y estación; del 1/1/2015 al 31/8/2026; líneas A, B, C, D, E, H y Premetro (desde 2025).
- **Calidad general:** sin nulos ni duplicados (sección 6.3).
- **Total contra suma de categorías:** solo difiere en junio de 2026, en las validadoras a bordo del Premetro (`COCHEPM`); viene del archivo original y es despreciable (sección 7).
- **Faltantes por estación:**(sección 8). Plaza de Mayo no tiene datos en 2020.
- **Decisiones para el análisis:** usar `total` para contar pasajeros.

## 10. Uso de inteligencia artificial

**Herramienta:** Claude (Anthropic), en una conversación de chat. La IA escribió el código a partir de los prompts de abajo; el grupo lo ejecutó en Colab y revisó las salidas. Cada explicación de este notebook indica qué prompt originó la celda que le sigue.

Los prompts se transcriben tal como se escribieron.

**Prompt E1 — Base para explorar**
> «ahora quiero que me hagas lo siguiente para darme la base para yo poder jugar con los datos y observarlos, armame un notebook que agarre los arhcivos rpocesados de github y los unifique en un solo datagrame para que yo peuda empezar a describirlos y analizarlos por mi cuenta»

*Qué aportó:* secciones 1 a 6.

**Prompt E2 — Total contra suma de categorías**
> [el grupo escribió y corrió un código que comparaba `total` con la suma de las tres categorías y encontró 275 filas que no cerraban]
> «esto podria provenir de cuando hicioms la limpieza de datos?»

*Qué aportó:* la búsqueda de dónde estaban las diferencias y la verificación en el archivo original (sección 7).

**Prompt E3 — Sumarlo a este notebook**
> «tal vez deberia sumar este analsiis en el notebook de exploracion de datos, y poner los resultados de este analisis en el mp file que crea no?»

*Qué aportó:* la organización de la sección 7.

**Prompt E4 — Notebook según la consigna y faltantes por estación**
> «okey, ahora armame el notebook de exploraicon de datos de acuerdo a lo que pide la consigna, de lo que te pedi que revises de si las columnas suman al total, y esto que notaste del fantante de plaza de mayo y otras estaicones en algunos años»
> «ya me habias armado un notebook que tenia este tabajo,peor agregale esto que estuvimos notado, y elimina lo que viene desde la aprte 7 de tablas de apoyo»

*Qué aportó:* la sección 8 y la versión final de este notebook.

**Qué revisó el grupo**
- [Corrimos el notebook completo y revisamos que no hubiera nulos ni duplicados.]
- [Detectamos las filas donde el total no coincidía con la suma de las categorías y comprobamos en el archivo original que el problema venía de ahí.]
- [Revisamos la tabla de faltantes por estación y el detalle mes a mes de Plaza de Mayo y Agüero.]